# Group 3 — v3 Embeddings (Reconstruction Loss, No-Log Target)

In [ ]:
import sys
import os
from pathlib import Path

sys.path.insert(0, os.path.abspath('../..'))

import pandas as pd

from src.models.fix_embeddings import FixedGNNConfig, build_fixed_pooled_dataset
from src.models.embeddings import Node2VecConfig

PROJECT_ROOT = Path().resolve().parents[1]
TARGET_DIR   = PROJECT_ROOT / 'src' / 'datasets' / 'targets'
OUTPUT_DIR   = PROJECT_ROOT / 'src' / 'data' / 'embeddings'

pd.set_option('display.max_columns', 200)
print(f'Project root: {PROJECT_ROOT}')

## Configuration

In [ ]:
TARGET_COL           = 'systemic_risk_label'
INCLUDE_RAW_FEATURES = False
YEARS                = range(2016, 2024)
QUARTERS             = (1, 2, 3, 4)

cfg_graphsage_32  = FixedGNNConfig(hidden_dims=(256, 32),  dropout=0.3, lr=0.01, epochs=100, reconstruction_weight=1.0, link_weight=0.0, aggregation='mean', device='cpu')
cfg_graphsage_64  = FixedGNNConfig(hidden_dims=(256, 64),  dropout=0.3, lr=0.01, epochs=100, reconstruction_weight=1.0, link_weight=0.0, aggregation='mean', device='cpu')
cfg_graphsage_128 = FixedGNNConfig(hidden_dims=(256, 128), dropout=0.3, lr=0.01, epochs=100, reconstruction_weight=1.0, link_weight=0.0, aggregation='mean', device='cpu')

cfg_node2vec_32   = Node2VecConfig(embedding_dim=32,  walk_length=20, context_size=10, walks_per_node=10, num_negative_samples=1, batch_size=128, lr=0.01, epochs=100, device='cpu')
cfg_node2vec_64   = Node2VecConfig(embedding_dim=64,  walk_length=20, context_size=10, walks_per_node=10, num_negative_samples=1, batch_size=128, lr=0.01, epochs=100, device='cpu')
cfg_node2vec_128  = Node2VecConfig(embedding_dim=128, walk_length=20, context_size=10, walks_per_node=10, num_negative_samples=5, batch_size=128, lr=0.01, epochs=100, device='cpu')

OUTPUTS = {
    'graphsage_32':  OUTPUT_DIR / 'graphsage_fixed_32_srisk_nolog_dataset.parquet',
    'graphsage_64':  OUTPUT_DIR / 'graphsage_fixed_64_srisk_nolog_dataset.parquet',
    'graphsage_128': OUTPUT_DIR / 'graphsage_fixed_128_srisk_nolog_dataset.parquet',
    'node2vec_32':   OUTPUT_DIR / 'node2vec_fixed_32_srisk_nolog_dataset.parquet',
    'node2vec_64':   OUTPUT_DIR / 'node2vec_fixed_64_srisk_nolog_dataset.parquet',
    'node2vec_128':  OUTPUT_DIR / 'node2vec_fixed_128_srisk_nolog_dataset.parquet',
}

OUTPUTS

## GraphSAGE v3

In [ ]:
for cfg, key in [
    (cfg_graphsage_32,  'graphsage_32'),
    (cfg_graphsage_64,  'graphsage_64'),
    (cfg_graphsage_128, 'graphsage_128'),
]:
    df = build_fixed_pooled_dataset(
        config=cfg,
        years=YEARS,
        quarters=QUARTERS,
        target_col=TARGET_COL,
        include_raw_features=INCLUDE_RAW_FEATURES,
        target_dir=TARGET_DIR,
        output_path=OUTPUTS[key],
    )
    emb_cols = [c for c in df.columns if c.startswith('emb_')]
    print(f'{key:15s}  shape={df.shape}  emb_cols={len(emb_cols)}')

## Node2Vec v3

In [ ]:
for cfg, key in [
    (cfg_node2vec_32,  'node2vec_32'),
    (cfg_node2vec_64,  'node2vec_64'),
    (cfg_node2vec_128, 'node2vec_128'),
]:
    df = build_fixed_pooled_dataset(
        config=cfg,
        years=YEARS,
        quarters=QUARTERS,
        target_col=TARGET_COL,
        include_raw_features=INCLUDE_RAW_FEATURES,
        target_dir=TARGET_DIR,
        output_path=OUTPUTS[key],
    )
    emb_cols = [c for c in df.columns if c.startswith('emb_')]
    print(f'{key:15s}  shape={df.shape}  emb_cols={len(emb_cols)}')

## Output

In [ ]:
for name, path in OUTPUTS.items():
    print(f'{name:20s} -> {path.name}')